# Compare ALE / Coordinate / DiFuMo Models

Loads ALE comparison rows from Google Drive by default, with a local fallback. If the aggregate CSV is missing, it discovers per-run `comparison_row.json` files under the Drive runs folder.

## Setup

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/neurovlm_ale3dcnn')
LOCAL_ROOT = Path('.')
ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else LOCAL_ROOT
RUNS_DIR = ROOT / 'runs'
COMPARISON_FILE = RUNS_DIR / 'ale_model_comparison.csv'
print('Using root:', ROOT)
print('Comparison CSV:', COMPARISON_FILE)


## Load Results

In [ ]:
def load_comparison_rows(runs_dir, comparison_file):
    if comparison_file.exists():
        print(f'Loading aggregate CSV: {comparison_file}')
        return pd.read_csv(comparison_file)

    row_paths = sorted(runs_dir.glob('ale_3dcnn_*_*/comparison_row.json'))
    if not row_paths:
        row_paths = sorted(runs_dir.glob('**/comparison_row.json'))
    if not row_paths:
        raise FileNotFoundError(
            f'No comparison CSV or comparison_row.json files found under {runs_dir}. '
            'Run the training notebook first, or set DRIVE_ROOT/RUNS_DIR to the folder containing runs.'
        )
    print(f'Found {len(row_paths)} comparison_row.json files')
    rows = []
    for path in row_paths:
        with open(path) as f:
            row = json.load(f)
        row['_source_file'] = str(path)
        rows.append(row)
    return pd.DataFrame(rows)

df = load_comparison_rows(RUNS_DIR, COMPARISON_FILE)
if 'paper_recall_curve_auc' not in df.columns and 'full_recall_curve_auc' in df.columns:
    df['paper_recall_curve_auc'] = df['full_recall_curve_auc']
if 'full_recall_curve_auc_k1_to_N' not in df.columns and 'paper_recall_curve_auc' in df.columns:
    df['full_recall_curve_auc_k1_to_N'] = df['paper_recall_curve_auc']
df.tail()


## Optional Manual Baselines

In [ ]:
# Add rows from existing NeuroVLM/CoordGNN/CoordDeepSet/DiFuMo GAT runs when you have them.
MANUAL_BASELINES = [
    # {'model_name': 'NeuroVLM MLP', 'preprocessing_type': 'difumo_flatmap', 'uses_difumo': 'yes', 'atlas_free': 'no',
    #  'input_shape': '(28542,)', 'number_of_parameters': None, 'training_time_per_epoch': None, 'peak_vram': None,
    #  'paper_recall_curve_auc': 0.81, 'recall@1': None, 'recall@5': None, 'recall@10': None, 'recall@50': None,
    #  'mrr': None, 'median_rank': None, 'random_recall@10': None, 'checkpoint_path': '', 'config_path': ''},
]
if MANUAL_BASELINES:
    df = pd.concat([df, pd.DataFrame(MANUAL_BASELINES)], ignore_index=True)
df


## Main Table

In [ ]:
metric_col = "paper_recall_curve_auc"
cols = [
    'model_name', 'preprocessing_type', 'uses_difumo', 'atlas_free', 'input_shape',
    'number_of_parameters', 'training_time_per_epoch', 'peak_vram',
    'paper_recall_curve_auc', 'full_recall_curve_auc_k1_to_N',
    'recall@1', 'recall@5', 'recall@10', 'recall@50',
    'mrr', 'median_rank', 'random_recall@10', 'checkpoint_path', 'config_path'
]
available_cols = [c for c in cols if c in df.columns]
table = df[available_cols].copy()
table = table.sort_values(metric_col, ascending=False, na_position="last")
table


## Retrieval AUC Plot

In [ ]:
plot_df = table.dropna(subset=[metric_col]).copy()
labels = plot_df['model_name'].astype(str) + ' / ' + plot_df['preprocessing_type'].astype(str)
fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_df))))
ax.barh(labels, plot_df[metric_col])
ax.invert_yaxis()
ax.set_xlabel('paper recall-curve AUC, k=1..N')
ax.set_title('Paper-Style Full Recall Curve AUC')
plt.tight_layout()


## Fixed Top-k Plot

In [ ]:
topk_cols = [c for c in ['recall@1', 'recall@5', 'recall@10', 'recall@50'] if c in plot_df.columns]
fig, ax = plt.subplots(figsize=(9, 4.5))
for col in topk_cols:
    ax.plot(labels, plot_df[col], marker="o", label=col)
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel('recall')
ax.set_title('Fixed Top-k Retrieval')
ax.legend()
plt.tight_layout()


## Interpretation Template

- If DiFuMo-compatible ALE3DCNN beats the existing MLP, spatial inductive bias helped.
- If DiFuMo-compatible ALE3DCNN does not beat the MLP, the MLP may already be sufficient for that representation.
- If atlas-free ALE3DCNN beats DiFuMo-compatible ALE3DCNN, the atlas/DiFuMo bottleneck may be limiting performance.
- If atlas-free ALEFlatMLP beats atlas-free ALE3DCNN, preprocessing mattered more than CNN architecture.
- If atlas-free ALE3DCNN beats atlas-free ALEFlatMLP, both preprocessing and spatial CNN architecture helped.
- If none beat the MLP, the limiting factor may be coordinate-derived proxy noise, text-pairing noise, or the training objective.
